# Lesson 18 — 参考实现

先完成主 Notebook 的尝试，再把本格与自己的实现逐行比较。

In [ ]:
# 运行支架：只执行一次。你写的网络逻辑仍是 C++，不需要学习 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))
from network_lab import register
register()


In [ ]:
%%network_lab l18-solution mqtt exercise pass
#include "academy/net/mqtt.hpp"
#include <iostream>

int main() {
    try {
        auto client = academy::net::MqttClient::connect("127.0.0.1", academy::net::env_port("ACADEMY_MQTT_PORT"), "desk-reminder");
        client.publish("desk/reminder/state", "offline");
        client.subscribe("desk/reminder/cmd");
        const auto command = client.receive_publish();
        const bool reminder_on = command.topic == "desk/reminder/cmd" && command.payload == "toggle";
        client.publish("desk/reminder/state", reminder_on ? "on" : "off");
        std::cout << "state=" << (reminder_on ? "on" : "off") << '\n';
        return reminder_on ? 0 : 1;
    } catch (const std::exception& error) {
        std::cerr << error.what() << '\n';
        return 1;
    }
}


此实现通过真实 localhost 契约。关键约束：订阅 `desk/reminder/cmd` 后用 `receive_publish()` 读取 Broker 的真实 PUBLISH packet。